In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("..")

In [3]:
from IPython.display import clear_output
from src.dataset_loaders import load_vectors, get_samplers
from src.utils import get_pca_models
from src import utils
from src.train import train_discrete
import wandb
from torch.utils.data import TensorDataset, DataLoader
import yaml
import numpy as np
import random
import pickle
import gc

## 1. Parameters.

Possible ```DATASET_NAME``` values are: ```twitter```, ```wiki-gigaword```, ```bone_marrow``` and ```muse```

In [4]:
DATASET_NAME = 'twitter'

SOURCE_DIM   = 100  
TARGET_DIM   = 50

EMB_TYPE_SOURCE = 'glove'
EMB_TYPE_TARGET = 'glove'

#SOURCE_LANG     = 'en'
#TARGET_LANG     = 'es'

MAX_ITERS    = 500
#VS           = 200000

In [5]:

METHOD_NAME  = 'FlowGW'
DEVICE       = 'cpu'

ALPHA_INIT    = 1.0
SEED_INIT     = 43
COST_DISCRETE = 'cosine'

config = {'dataset':dict(DATASET_NAME     = DATASET_NAME,
                         DEVICE           = DEVICE,
                         SOURCE_DIM       = SOURCE_DIM,
                         TARGET_DIM       = TARGET_DIM,
                         EMB_TYPE_SOURCE  = EMB_TYPE_SOURCE,
                         EMB_TYPE_TARGET  = EMB_TYPE_TARGET,
                        # VS               = VS,
                         #SOURCE_LANG      = SOURCE_LANG,
                         #TARGET_LANG      = TARGET_LANG,
                         
                         N_MAX_SAMPLES    = 400000, #set to 6667 to get N_train=3K
                         N_TRAIN_SAMPLES  = 6000, #We used 6000 for the others
                         N_TEST_SAMPLES   = 512,
                         N_EVAL           = 4,
                         ALPHA            = ALPHA_INIT, 
                         SEED             = SEED_INIT,
                         NORMALIZE_VECS   = False,
                         SHUFFLE          = True
                         ),
          
          'training':dict(TRAIN_TYPE           = 'discrete',
                          METHOD_NAME          = METHOD_NAME,
                          MAX_ITERS            = MAX_ITERS,
                          COST_DISCRETE        = COST_DISCRETE,
                          ),

          #===============================RegGW===============================
          #'model_specific':dict(HIDDEN_SIZES_MLP = [512, 256, 256],
          #                      EPS_FIT          = 0.01,
          #                      EPS_REG          = 0.001,
          #                      LAMBDA           = 1,
          #                      MOVER_LR         = 1e-4
          #                     ),

          #===============================FlowGW===============================
          'model_specific':dict(HIDDEN_SIZES_MLP = [1024, 1024, 1024, 1024],
                                EPS              = 1e-4,
                                N_FREQ           = 128,
                                MOVER_LR         = 1e-4,
                                )
          
          #===============================AlignGW===============================
          #===============================StructuredGW===============================
          
          #'model_specific':dict(EPS = 1e-4,
          #                     ),
         }


## 2. Loading dataset.

In [6]:
dataset_path = '../datasets'
sys.path.append(dataset_path)

source_vectors, target_vectors = load_vectors(dataset_path, config)

print(source_vectors.shape)
print(target_vectors.shape)


Loading twitter_glove_100 to source...
Loading twitter_glove_50 to target...
torch.Size([400000, 100])
torch.Size([400000, 50])


## 3. Training.

In [ ]:
import os
#os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

n_repeats = 5

wandb_report = True
if wandb_report is False:
    wandb_mode = 'disabled'
else:
    wandb_mode = 'online'

N            = config['dataset']['N_MAX_SAMPLES']//1000
TRAIN_TYPE   = config['training']['TRAIN_TYPE']
project_name = f'{METHOD_NAME}_{DATASET_NAME}_{SOURCE_DIM}({EMB_TYPE_SOURCE})->{TARGET_DIM}({EMB_TYPE_TARGET})_{N}K_{n_repeats}reps_fixed_marginals'
#project_name = f'{METHOD_NAME}_{DATASET_NAME}_{SOURCE_DIM}({SOURCE_LANG}_{EMB_TYPE_SOURCE})->{TARGET_DIM}({TARGET_LANG}_{EMB_TYPE_TARGET})_{N}K_{n_repeats}reps_vs({VS//1000}K)_final'

metrics_names = ['Top@1', 'Top@5', 'Top@10', 'cossim_gt', 'inner_gw', 'foscttm', 'distortion', 'mmd', 'bw_uvp', 'sinkhorn_divergence']

_, _, _, _, test_sampler = get_samplers(config, source_vectors, target_vectors)      

alpha_values = [0.0, 0.4, 0.7, 1.0][::-1]

metrics_out = {str(np.round(alpha, 1)):[] for alpha in alpha_values}

for ALPHA in alpha_values:
    
    config['dataset']['ALPHA'] = ALPHA 
    
        
    print('================================')
    print(f'Experiment for ALPHA={ALPHA}')
    print('================================')
    
    for ix in range(n_repeats):
        
        if wandb_report:
            exp_name = f'ALPHA_{np.round(ALPHA, 1)}_repeat_{ix}'
            wandb.init(name=exp_name, config=config, project=project_name, mode=wandb_mode)
            
        SEED = random.randint(0, 10000)
        config['dataset']['SEED'] = SEED
        print('Seed: ', SEED)
        
        source_vectors, target_vectors, train_source_sampler, train_target_sampler, _ = get_samplers(config, source_vectors, target_vectors) 
        
        trained_class, metrics_dict = train_discrete(train_source_sampler, train_target_sampler,
                                                     test_sampler, 
                                                     metrics_names, target_vectors,
                                                     config,
                                                     wandb_report=wandb_report,
                                                     axis_lims=None, report_every=10)
        
        metrics_out[str(np.round(ALPHA, 1))].append(metrics_dict)

        with open(f'results_{TRAIN_TYPE}_final/{project_name}_00.pkl', 'wb') as f:
            pickle.dump(metrics_out, f)

        gc.collect()

Source pairs...
3000
tensor([ 20212, 149990, 365022,  ..., 198499, 291696, 328752],
       dtype=torch.int32)
Target pairs...
3000
tensor([ 20212, 149990, 365022,  ..., 198499, 291696, 328752],
       dtype=torch.int32)
Experiment for ALPHA=1.0


wandb: Currently logged in as: xavier13091994 (entropic_gw). Use `wandb login --relogin` to force relogin


Seed:  3017
Source pairs...
3000
tensor([347301, 226521,  26169,  ..., 281842,  43073, 228636],
       dtype=torch.int32)
Target pairs...
3000
tensor([347301, 226521,  26169,  ..., 281842,  43073, 228636],
       dtype=torch.int32)
GT distortion: 0.03690521419048309
Random distortion (seed=3017): 0.03700640797615051


2025-04-22 11:08:31.152338: W external/xla/xla/service/gpu/nvptx_compiler.cc:765] The NVIDIA driver's CUDA version is 12.3 which is older than the ptxas CUDA version (12.5.40). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


  0%|          | 0/500 [00:00<?, ?it/s]

/trinity/home/xavier.aramayo/miniconda3/envs/jax/lib/python3.10/site-packages/jax/_src/core.py:678: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/trinity/home/xavier.aramayo/miniconda3/envs/jax/lib/python3.10/site-packages/jax/_src/core.py:678: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(


here1
here2
here3
